<a href="https://colab.research.google.com/github/descartesmbogning/phage-therapy-funding/blob/main/Patent_TopicModeling_phage_funding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# data import and preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!pip install pyLDAvis==3.4.0

In [ ]:
import pyLDAvis
import pyLDAvis.sklearn

In [ ]:

import spacy
import codecs
import re
from wordcloud import STOPWORDS

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn import decomposition
from nltk.stem.porter import PorterStemmer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_excel ("/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data/DATA_CLEAN_to_used.xlsx")
pd.set_option('display.max_columns', None)
print(len(df))
df.head()

In [ ]:
## Convert 'Publication Year' to integers
df['Publication Year'] = df['Start Year'].astype(int)

# Filter based on 'Publication Year' column
#df = df[df['Publication Year'] >= 2003]
len(df)

In [ ]:
# df = df[(~pd.isnull(df['Abstract'])) & (~pd.isnull(df['Title']))]
# len(df)

In [ ]:
df = df.set_index(['Grant ID','Publication Year'])
df

In [ ]:
data = [str(df['Title translated'][i]) +  ' ' +str(df['Abstract translated'][i])  +  ' ' +str(df['Keywords'][i]) for i in df.index] #Title and Abstract
len(data)

In [ ]:
data

In [ ]:
!python -m spacy download en_core_web_md

In [ ]:
import en_core_web_md
nlp = en_core_web_md.load()

In [ ]:
import nltk
nltk.download('punkt')

from wordcloud import WordCloud, STOPWORDS
import plotly.express as px
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
nltk.download('stopwords')
s = set(STOPWORDS)
sw = stopwords.words('english')
sw += s
sw.append ('amp')

In [ ]:
from wordcloud import WordCloud, STOPWORDS
import plotly.express as px
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Downloads
nltk.download('wordnet')
nltk.download('stopwords')

# Initialize your stopwords set with wordcloud's stopwords
s = set(STOPWORDS)

# Get stopwords for all languages in NLTK
languages = stopwords.fileids()
for language in languages:
    s.update(stopwords.words(language))

# Add any additional stopwords if needed
s.add('amp')

# If you really need the resulting stopwords as a list:
sw = list(s)


In [ ]:
# Read the texts and remove the stopwords :
#Build the text list:
txt = []
idx = []
t1 = ''
t2 = []
for i in range(len(data)):
        t1 = str(data[i]).lower()
 #       t1 = re.sub ('#','',t1)
        #t1 = re.sub ('.','',t1)
 #       t1 = re.sub (',','',t1)
 #       t1 = re.sub ("'",'',t1)
#        t1 = t1.replace('.','')
        t2 = t1.split()
        t1 = ''
        for item2 in t2:
            if item2 not in sw:
                t1 += item2
                t1 += ' '
        if t1[:-1] not in txt and t1[:-1] != '':
            txt.append (t1[:-1])
            idx.append (i)
len (txt)

In [ ]:
#pip install --upgrade ipykernel

In [ ]:
idx

In [ ]:
#txt

In [ ]:
print(type(txt))

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud, STOPWORDS

# Assume df is your original DataFrame
# Remove rows where either 'Abstract' or 'Title' is null
# df = df.dropna(subset=['Abstract', 'Title'])

# Generate the data list combining 'Title' and 'Abstract'
data = [str(df['Title translated'].iloc[i]) + ' ' + str(df['Abstract translated'].iloc[i])  + ' ' + str(df['Keywords'].iloc[i]) for i in range(len(df))]

#

## Download necessary NLTK resources
#nltk.download('wordnet')
#nltk.download('stopwords')

## Generate a list of stopwords
#s = set(STOPWORDS)
#sw = set(stopwords.words('english'))
#sw |= s
#sw.add('amp')

# Initialize a list to store cleaned texts
cleaned_texts = []

# Text preprocessing
for text in data:
    words = text.lower().split()
    cleaned_words = [word for word in words if word not in sw]
    cleaned_text = ' '.join(cleaned_words)
    cleaned_texts.append(cleaned_text)

# Add the cleaned texts back to the original DataFrame
df['Cleaned_Text'] = cleaned_texts
len(df)

In [ ]:
# Drop rows where 'Cleaned_Text' is null
df_filtered = df.dropna(subset=['Cleaned_Text'])

# Keep only the rows with unique 'Cleaned_Text'
df_filtered = df_filtered.drop_duplicates(subset=['Cleaned_Text'])
df_filtered

In [ ]:
text = df['Cleaned_Text'].tolist()
text

In [ ]:
def lemmatization (texts, allowed_postags=['NOUN','ADJ','VERB']):
    texts_out = []
    for text in texts:
        doc = nlp(text)
        new_text = []
        for token in doc:
            if token.pos_ in allowed_postags:
                new_text.append (token.lemma_)
        final = " ".join(new_text)
        texts_out.append (final)
    return (texts_out)

In [ ]:
txt = lemmatization (txt)

In [ ]:
# Tokenization
def tokenize (texts):
    tokens = [word for word in nltk.word_tokenize(texts) if len(word) > 2]
    return tokens

In [ ]:
tok = []
for item in txt:
  tok.append (tokenize (item))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ... [your previous code to create the set 's' of stopwords] ...

# Using the set 's' as stopwords for TfidfVectorizer
vectorizer_tf = TfidfVectorizer(tokenizer=tokenize, stop_words=list(s), max_df=0.75, min_df=50, use_idf=False, norm=None)


In [ ]:
#vectorizer_tf = TfidfVectorizer (tokenizer=tokenize, stop_words='english', max_df=0.75, min_df=50,  use_idf=False, norm=None) #max_features=10000,

In [ ]:
#df

In [ ]:
df1 = pd.DataFrame(data={'Text':txt})
tf_vectors = vectorizer_tf.fit_transform (df1['Text'])

In [ ]:
#data.to_frame()

In [ ]:
df1

In [ ]:
import pickle
with open('/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/vectorizer_tf_phage_funding.pk', 'wb') as f:
     pickle.dump(vectorizer_tf, f)

In [ ]:
import scipy.sparse
scipy.sparse.save_npz('/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/tf_vectors_phage_funding.npz', tf_vectors)

In [ ]:
# Save this model with 20 topics:
import pickle
with open("/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/vectorizer_tf_phage_funding.pk","rb") as f:
    vectorizer_tf = pickle.load (f)

import scipy.sparse
tf_vectors = scipy.sparse.load_npz('/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/tf_vectors_phage_funding.npz')

# Finding the Best Number of Topics

In [ ]:
!pip install tmtoolkit

In [ ]:
import tmtoolkit
from tmtoolkit.topicmod.evaluate import metric_coherence_gensim
from statistics import mean
from sklearn.metrics import jaccard_score

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import decomposition
from sklearn.metrics import jaccard_score
import numpy as np

# Assuming tf_vectors, vectorizer_tf, and tok are defined earlier in your code

results = pd.DataFrame(columns=['num_topic', 'Coherence', 'Jaccard', 'Perplexity'])

for topic_num in range(2, 20):
    lda = decomposition.LatentDirichletAllocation(n_components=topic_num, learning_method='online', learning_offset=50, n_jobs=-1, random_state=111)

    w1 = lda.fit_transform(tf_vectors)
    ppx = lda.perplexity(w1, sub_sampling=False)
    coh = mean(metric_coherence_gensim(measure='c_v', top_n=30, topic_word_distrib=lda.components_, vocab=np.array([x for x in vectorizer_tf.vocabulary_.keys()]), texts=tok))

    l = len(lda.components_)
    n = 0
    js = 0
    for i in range(0, l-1):
        topic1 = [1 if x > 1 else 0 for x in lda.components_[i]]
        for j in range(i+1, l):
            topic2 = [1 if x > 1 else 0 for x in lda.components_[j]]
            js += jaccard_score(topic1, topic2)
            n += 1
    js /= n

    results = results.append({'num_topic': topic_num,
                              'Coherence': coh,
                              'Jaccard': js,
                              'Perplexity': ppx
                              }, ignore_index=True)

# Print the DataFrame
print(results)

In [ ]:
# Print the DataFrame
print(results)

In [ ]:
# Plotting
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Coherence and Jaccard on the primary y-axis
ax1.plot(results['num_topic'], results['Coherence'], label='Coherence', color='blue')
ax1.plot(results['num_topic'], results['Jaccard'], label='Jaccard', color='green')
ax1.set_xlabel('Number of Topics')
ax1.set_ylabel('Coherence & Jaccard Scores')

# Create a second y-axis for Perplexity
ax2 = ax1.twinx()
ax2.plot(results['num_topic'], results['Perplexity'], label='Perplexity', color='red')
ax2.set_ylabel('Perplexity')

# Adding legends
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title('LDA Model Metrics with Dual Axis for Perplexity')
plt.show()

In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
plt.plot(results['num_topic'], results['Coherence'], label='Coherence')
#plt.plot(results['num_topic'], results['Jaccard'], label='Jaccard')
#plt.plot(results['num_topic'], results['Perplexity'], label='Perplexity')
plt.xlabel('Number of Topics')
plt.ylabel('Scores')
plt.title('LDA Model Metrics')
plt.grid(True)
plt.legend()
plt.show()



```
coh0 = 0
js0 = 100
ppx0 = 100
num = 1
for topic_num in range (5, 100):
  lda = decomposition.LatentDirichletAllocation (n_components=topic_num, learning_method='online', learning_offset=50, n_jobs=-1, random_state=111)
  # n_components is the number of topics

  w1 = lda.fit_transform (tf_vectors)
  #h1 = lda.components_

  ppx = lda.perplexity(w1,  sub_sampling=False)

  coh = mean (metric_coherence_gensim(measure='c_v', top_n=30, topic_word_distrib=lda.components_, vocab=np.array([x for x in vectorizer_tf.vocabulary_.keys()]), texts=tok))

  l = len (lda.components_)
  n = 0
  js = 0
  for i in range(0, l-1):
    topic1 = [1 if x>1 else 0 for x in lda.components_[i]]
    for j in range(i+1, l):
      topic2 = [1 if x>1 else 0 for x in lda.components_[j]]
      js += jaccard_score (topic1, topic2)
      n += 1
  js /= n

  if js > js0 and coh < coh0 and ppx > ppx0:
    pass
  else:
    num = topic_num
    js0 = js
    coh0 = coh
    ppx0 = ppx

  print ('num_topic: ', topic_num, 'Coherence: ', coh, 'Jaccard: ', js, 'Perplexity: ', ppx)
#print ('BEST: ', 'num_topic: ', num, 'Coherence: ', coh0, 'Jaccard: ', js0, 'Perplexity: ', ppx0)
```



In [ ]:
XXXXXXXXXXXXXXXXXXXXXXXXX

# Topic Modeling

In [ ]:
import pandas as pd

def get_smallest_topic_with_max_metric(df, metric):
    # Find the max value in the metric column
    max_value = df[metric].max()

    # Filter the DataFrame for rows with this max value
    df_max_value = df[df[metric] == max_value]

    # Get the smallest 'num_topic' in these rows
    topic_num = df_max_value['num_topic'].min()

    return topic_num


In [ ]:
n_topics = int(get_smallest_topic_with_max_metric(results, 'Coherence'))

n_topics

In [ ]:
lda = decomposition.LatentDirichletAllocation (n_components=n_topics, learning_method='online', learning_offset=50, n_jobs=-1, random_state=111)
# n_components is the number of topics

w1 = lda.fit_transform (tf_vectors)
h1 = lda.components_

In [ ]:
import pickle
with open(f"/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/lda_phage_funding_{n_topics}.pk","wb") as f1:
    pickle.dump (lda, f1)

In [ ]:
import pickle
lda = decomposition.LatentDirichletAllocation (n_components=n_topics, learning_method='online', learning_offset=50, n_jobs=-1, random_state=111)
with open(f"/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/lda_phage_funding_{n_topics}.pk","rb") as f1:
    lda = pickle.load (f1)

In [ ]:
#pyLDAvis.enable_notebook()
#pyLDAvis.sklearn.prepare(lda, tf_vectors, vectorizer_tf, mds='tsne')

What did I do to make this work?

I installed pyLDAvis, but didn't import it!

I went to usr/local/lib/python3.10/dist-packages/pyLDAvis/sklearn.py
and changed get_feature_names() to get_feature_names_out()

saved it!
Then imported pyLDAvis and pyLDAvis.sklearn

In [ ]:
w1 = lda.transform (tf_vectors)

In [ ]:
w1.shape

In [ ]:
text = []
topic = []
probabilit = []
for i in range(len(idx)):
  text.append (str(data[idx[i]]))
  t = np.argmax (w1[i,:])
  topic.append (t)
  probabilit.append (w1[i,t])
probabilities = pd.DataFrame (data= {'Text':text,'Topic':topic,'Probability':probabilit})

In [ ]:
print('Topic:','Number:')
for i in range(n_topics):
  print('topic',i,'    ',list(probabilities['Topic']).count(i))

In [ ]:
# topic similarity matrix

In [ ]:
## jaccard_similarity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import LatentDirichletAllocation

def jaccard_similarity(vector1, vector2, threshold=0.01):
    set1 = set(np.where(vector1 > threshold)[0])
    set2 = set(np.where(vector2 > threshold)[0])
    intersection = len(set1.intersection(set2))
    union = len(set1) + len(set2) - intersection
    if union == 0:
        return 0
    else:
        return intersection / union


# Assuming lda is your trained LDA model
# Normalize the topic-word distribution
topic_word_distributions = lda.components_ / lda.components_.sum(axis=1)[:, np.newaxis]

# Number of topics
n_topics = lda.components_.shape[0]

# Create Jaccard Similarity matrix
jaccard_similarity_matrix = np.zeros((n_topics, n_topics))

for i in range(n_topics):
    for j in range(n_topics):
        jaccard_similarity_matrix[i, j] = jaccard_similarity(topic_word_distributions[i, :], topic_word_distributions[j, :], threshold=0.01)


# Plot Heatmap
plt.figure(figsize=(25, 15))
sns.set(font_scale=1)
sns.heatmap(
    jaccard_similarity_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    cbar=True,
    square=True,
    annot_kws={"size": 12}
)
plt.title("Topic Similarity Matrix based on Jaccard Similarity")
plt.xlabel("Topic ID")
plt.ylabel("Topic ID")

# Save the figure
plt.savefig(f'/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/topic_similarity_matrix_jaccard_phage_funding.png', dpi=600)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.decomposition import LatentDirichletAllocation

# Define Jaccard Similarity function
def jaccard_similarity(vector1, vector2, threshold=0.01):
    set1 = set(np.where(vector1 > threshold)[0])
    set2 = set(np.where(vector2 > threshold)[0])
    intersection = len(set1.intersection(set2))
    union = len(set1) + len(set2) - intersection
    if union == 0:
        return 0
    else:
        return intersection / union


# Assuming lda is your trained LDA model
n_topics = lda.components_.shape[0]
topic_word_distributions = lda.components_ / lda.components_.sum(axis=1)[:, np.newaxis]

# Create Jaccard Similarity matrix
jaccard_similarity_matrix = np.zeros((n_topics, n_topics))

for i in range(n_topics):
    for j in range(n_topics):
        jaccard_similarity_matrix[i, j] = jaccard_similarity(topic_word_distributions[i, :], topic_word_distributions[j, :], threshold=0.01)

# Perform hierarchical clustering
linkage_matrix = linkage(1 - jaccard_similarity_matrix, method="average")

# Plot the dendrogram
plt.figure(figsize=(15, 10))
dendrogram(linkage_matrix, color_threshold=0.5)
plt.title("Hierarchical Clustering of Topics based on Jaccard Similarity")
plt.xlabel("Topic ID")
plt.ylabel("Distance")
plt.tight_layout()

# Save the figure in high resolution
plt.savefig(f'/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/topic_clustering_dendrogram_jaccard_phage_funding.png', dpi=600)

plt.show()


In [ ]:
# get_top_keywords

In [ ]:
def get_top_keywords(lda_model, vectorizer, n_words=50):
    terms = vectorizer.get_feature_names_out()
    topic_keywords = []

    for topic_idx, topic in enumerate(lda.components_):
        top_keywords_idx = topic.argsort()[::-1][:n_words]
        top_keywords = [(terms[i], topic[i]) for i in top_keywords_idx]
        topic_keywords.append(top_keywords)

    return topic_keywords

top_keywords = get_top_keywords(lda, vectorizer_tf, n_words=50)

# To display the topics with their keywords and scores
with open(f"/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/top_keywords_phage_funding.txt", "w") as f:
    for idx, keywords in enumerate(top_keywords):
        f.write(f"Topic {idx}:\n")
        print(f"Topic {idx}:")
        for word, score in keywords:
            f.write(f"{word}, ")
            print(f"{word}", end=", ")
        f.write("\n\n")
        print("\n")

In [ ]:
# Emergent topics

In [ ]:
from wordcloud import WordCloud
import numpy as np
import matplotlib.pyplot as plt

# Word Clouds for topics
def plot_wordcloud(lda, topic_id, ax, n_words=50):
    terms = vectorizer_tf.get_feature_names_out()
    word_probs = {terms[i]: prob for i, prob in enumerate(lda.components_[topic_id])}
    wc = WordCloud(background_color='white', max_words=n_words).generate_from_frequencies(word_probs)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f"Topic {topic_id}", fontweight='bold')  # Bold title for each subplot

# Assuming n_topics to be the number of topics you've chosen
n_topics = n_topics  # Make sure this matches the number of topics in lda_model

# Calculate the number of rows needed
n_rows = int(np.ceil(n_topics / 5))

fig, axes = plt.subplots(n_rows, 5, figsize=(20, 4 * n_rows))

# Flatten the array and truncate it to match the number of topics
axes_flat = axes.flatten()[:n_topics]

for ax, topic_id in zip(axes_flat, range(n_topics)):
    plot_wordcloud(lda, topic_id, ax)

# Remove extra subplots
for idx in range(n_topics, len(axes.flatten())):
    fig.delaxes(axes.flatten()[idx])

plt.suptitle("Word Clouds for LDA Topics", fontweight='bold')  # Bold main title
plt.tight_layout()
plt.savefig(f'/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/wordclouds_phage_funding.png', dpi=600)  # Use raw string for path
plt.show()


In [ ]:
# Attach Discovered Topic Labels to Original scholar publications

In [ ]:
final_topics = lda.transform(tf_vectors)
final_topics.shape

In [ ]:
df_filtered["Topic N°"] = final_topics.argmax(axis=1)
df_filtered.head()

In [ ]:
# Save data to CSV and Pickle formats
df_filtered.to_csv(f'/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/topic_probability_phage_funding.csv', index=False)
df_filtered.to_pickle(f'/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/topic_probability_phage_funding.pkl')

df_filtered.head()

In [ ]:
df_filtered

In [ ]:
df_filtered['Topic N°'].value_counts()

In [ ]:
# Frequency Counting

In [ ]:
# Step 1: Group data by 'Publication Year' and 'topic', then count occurrences
topic_counts_by_year = df_filtered.reset_index().groupby(['Publication Year', 'Topic N°']).size().unstack(fill_value=0)

# Step 2: Calculate the total count of each topic
total_counts = topic_counts_by_year.sum()

# Step 3: Sort columns by total count
sorted_columns = total_counts.sort_values(ascending=False).index
topic_counts_by_year = topic_counts_by_year[sorted_columns]

# Step 4: Add a row with the total count of each topic
topic_counts_by_year.loc['Total'] = total_counts.loc[sorted_columns]

# Save the DataFrame to CSV and pickle files
topic_counts_by_year.to_csv(f'/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/topic_counts_by_year_phage_funding.csv')
topic_counts_by_year.to_pickle(f'/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/topic_counts_by_year_phage_funding.pkl')

# This will give you a DataFrame where each row is a year (plus a 'Total' row at the bottom),
# and each column is the count of documents associated with a topic for that year (and the total count).

# Display the updated DataFrame
topic_counts_by_year

In [ ]:
topic_counts_by_year.shape

In [ ]:
# Evolution Curve for Each Topic

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Read the DataFrame from the CSV file (assuming it's stored as 'topic_counts_by_year.csv')
# Replace this line with your actual data if it's not in a CSV
#topic_counts_by_year = pd.read_csv('topic_counts_by_year.csv', index_col='Publication Year')

# Remove the 'Total' row if it exists
if 'Total' in topic_counts_by_year.index:
    topic_counts_by_year = topic_counts_by_year.drop(index=['Total'])

# Sort columns by topic name
topic_counts_by_year = topic_counts_by_year.sort_index(axis=1)

# Define the number of topics (make sure this matches your data)
n_topics = len(topic_counts_by_year.columns)

# Calculate the number of rows needed for subplots
n_rows = (n_topics + 4) // 5  # +4 for rounding up

# Create a grid of subplots
fig, axes = plt.subplots(n_rows, 5, figsize=(20, n_rows * 3.5))

# Flatten the array to easily iterate over it
axes_flat = axes.flatten()

# Loop through each topic and plot its curve on a separate subplot
for i, topic in enumerate(topic_counts_by_year.columns):
    axes_flat[i].plot(
        topic_counts_by_year.index,
        topic_counts_by_year[topic]
    )
    axes_flat[i].set_title(f'Topic {topic}')
    axes_flat[i].set_xlabel('Year')
    axes_flat[i].set_ylabel('Frequency')
    axes_flat[i].grid(True, which='both', linestyle='--', linewidth=0.5)

# Remove extra subplots
for i in range(n_topics, len(axes_flat)):
    fig.delaxes(axes_flat[i])

# Add a main title for the entire figure
fig.suptitle('Evolution Curve for Each Topic', fontsize=16, fontweight='bold')

# Save the figure
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Make room for the main title
plt.savefig(f'/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/topic_evolution_curves_subplot_ordered_phage_funding.png', dpi=600)

# Show the figure
plt.show()


In [ ]:
# Temporal Heatmaps: Display how topic proportions change over time.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Assume you already have topic_counts_by_year DataFrame

# Remove the 'Total' row if it exists
if 'Total' in topic_counts_by_year.index:
    topic_counts_by_year = topic_counts_by_year.drop(index=['Total'])

# Sort columns by topic name
topic_counts_by_year = topic_counts_by_year.sort_index(axis=1)

# Calculate the total count for each year (row-wise sum)
total_counts_by_year = topic_counts_by_year.sum(axis=1)

# Normalize frequencies to get proportions
proportions_by_year = topic_counts_by_year.divide(total_counts_by_year, axis=0)

# Create the heatmap
plt.figure(figsize=(25, 10))
sns.heatmap(proportions_by_year.fillna(0), annot=True, cmap="coolwarm", cbar=True, xticklabels=True, yticklabels=True)

# Add labels and title
plt.title('Temporal Heatmap of Topic Proportions')
plt.xlabel('Topics')
plt.ylabel('Publication Year')

# Save the figure
plt.tight_layout()
plt.savefig(f"/content/drive/Othercomputers/My laptop/D:/PROJECT/YORK UNIVERSITY/phage funding/data_preprocesse/temporal_heatmap_topic_proportions_phage_funding.png")  # Save the plot to a PNG file

# Show the plot
plt.show()
